## CESM2-WACCM and EN4 regridding
This notebook does the following:
- crop CMIP files to EPSG:3413 region of interest (immediately makes the files much more manageable) using function crop_cmip.py
- convert temperature and salinity into thermal forcing using function calc_cmip_TF.py
    - **Caution! `calc_cmip_TF.py` as provided in this repo includes a correction from cm to m depth units.** Check the units of your GCM file and adjust the function appropriately.  Check that your output is the correct order of magnitude using the plotting tools we provide.
- regrid CMIP output to regular grid using function regrid_cmip.py
- regrid EN4 output to the same regular grid using function regrid_en4.py

### Setup
Note that this implementation assumes you have files stored in time slices.  You will comment/uncomment below and run the subsequent cells again for each slice.  This can certainly be updated in a for-loop or other restructuring for high performance computing.

In [ ]:
# region of interest in EPSG:3413 and format [xmin,xmax,ymin,ymax]
roi = [-7.2e5,9.6e5,-3.45e6,-0.57e6]

# linear freezing point parameters
l1 = -5.73e-2
l2 = 8.32e-2
l3 = -7.61e-4

# regular grid
hres = 40e3
vres = 50

# files to process
base_dir = '/Volumes/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM'
# historical: 1850-2014
# cmipTfile = base_dir + '/historical/Omon/thetao/thetao_Omon_CESM2-WACCM_historical_r1i1p1f1_gn_185001-201412.nc'
# cmipSfile = base_dir + '/historical/Omon/so/so_Omon_CESM2-WACCM_historical_r1i1p1f1_gn_185001-201412.nc'

# projection: 2015-2100
# cmipTfile = base_dir + '/ssp585/Omon/thetao/thetao_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_201501-210012.nc'
# cmipSfile = base_dir + '/ssp585/Omon/so/so_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_201501-210012.nc'

# 2101-2150
# cmipTfile = base_dir + '/ssp585/Omon/thetao/thetao_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_210101-215012.nc'
# cmipSfile = base_dir + '/ssp585/Omon/so/so_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_210101-215012.nc'

# 2151-2200
# cmipTfile = base_dir + '/ssp585/Omon/thetao/thetao_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_215101-220012.nc'
# cmipSfile = base_dir + '/ssp585/Omon/so/so_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_215101-220012.nc'

# 2201-2250
# cmipTfile = base_dir + '/ssp585/Omon/thetao/thetao_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_220101-225012.nc'
# cmipSfile = base_dir + '/ssp585/Omon/so/so_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_220101-225012.nc'

# 2251-2299
cmipTfile = base_dir + '/ssp585/Omon/thetao/thetao_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_225101-229912.nc'
cmipSfile = base_dir + '/ssp585/Omon/so/so_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_225101-229912.nc'

### Imports

In [ ]:
import numpy as np
from crop_cmip import crop_cmip
from calc_cmip_TF import calc_cmip_TF
from regrid_cmip import regrid_cmip

### Setup the regular grid

In [ ]:
xreg = np.arange(roi[0],roi[1]+hres,hres)
yreg = np.arange(roi[2],roi[3]+hres,hres)
[Xreg,Yreg] = np.meshgrid(xreg,yreg)
zreg = np.arange(0,2000+vres,vres)

### 1. Crop CMIP raw data to region of interest
Takes around 5 mins per 50 years of CMIP output

In [ ]:
# number of time slices to process at a time (120 seems about optimal)
n_chunk = 120

# do crop
crop_cmip(cmipTfile,cmipSfile,roi,n_chunk)

### 2. Calculate CMIP thermal forcing
Takes around 1 min per 50 years of CMIP output

In [ ]:
# files from previous stage of processing
cmipTfile_cropped = cmipTfile.replace('.nc','_cropped.nc')
cmipSfile_cropped = cmipSfile.replace('.nc','_cropped.nc')

# calculate TF
calc_cmip_TF(cmipTfile_cropped,cmipSfile_cropped,l1,l2,l3)

### 3. Put CMIP thermal forcing on regular grid

Takes around 10 mins per 50 years of CMIP output

In [ ]:
# file from previous stage of processing
cmipTFfile_cropped = cmipTfile.replace('thetao','TF').replace('.nc','_cropped.nc')

# do regrid
regrid_cmip(cmipTFfile_cropped,xreg,yreg,zreg)

### 3a. Illustrate regridding

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

# cmip gridding
origfile = '/Volumes/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/ssp585/Omon/TF/TF_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_201501-210012_cropped.nc'
dsorig = xr.open_dataset(origfile)
x = dsorig['x'].values
y = dsorig['y'].values
TF = dsorig['TF'].values

# regular gridding
gridfile = '/Volumes/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/ssp585/Omon/TF/TF_Omon_CESM2-WACCM_ssp585_r1i1p1f1_gn_201501-210012_cropped_regrid.nc'
dsgrid = xr.open_dataset(gridfile)
xgrid = dsgrid['x'].values
ygrid = dsgrid['y'].values
TFgrid = dsgrid['TF'].values

# plot
fig = plt.figure(figsize=(14,16))
plt.pcolormesh(xgrid,ygrid,TFgrid[0,0,:,:],edgecolors='w',linewidth=0.5)
plt.scatter(x,y,c=TF[0,0,:],s=10,edgecolors='w')
plt.axis('equal')
plt.title('40-km gridded and original resolution (circles) CESM2-WACCM TF at surface')
plt.colorbar()
plt.show()